In [6]:
#@title { vertical-output: true}
!git clone https://github.com/maxwell-petitjean/fpl_26.git
%cd fpl_26
!pip install -q -r requirements.txt

Cloning into 'fpl_26'...
remote: Enumerating objects: 113, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 113 (delta 36), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (113/113), 6.60 MiB | 5.21 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/fpl_26/fpl_26
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 54.5 MB/s eta 0:00:00


In [5]:
#@title { vertical-output: true}
import pandas as pd
from pathlib import Path

HISTORIC_PATH = Path("data/raw/historic")

seasons = sorted([
    p.name
    for p in HISTORIC_PATH.iterdir()
    if p.is_dir()
])

print("Seasons:", seasons)

Seasons: ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']


In [7]:
#@title { vertical-output: true }

frames = []

for season in seasons:
    path = HISTORIC_PATH / season / "players_raw.csv"

    df = pd.read_csv(path, low_memory=False)

    keep = [
        c for c in [
            "id",
            "code",
            "opta_code",
            "first_name",
            "second_name",
            "web_name",
            "team",
            "element_type"
        ]
        if c in df.columns
    ]

    df = df[keep].copy()
    df["season"] = season

    df = df.rename(columns={
        "id": "fpl_element_id",
        "code": "player_code",
        "team": "team_id",
        "element_type": "position_id"
    })

    frames.append(df)

players = pd.concat(frames, ignore_index=True)

print(f"Rows: {len(players):,}")
print(f"Unique player codes: {players['player_code'].nunique():,}")
players.head()

Rows: 4,025
Unique player codes: 1,797


,fpl_element_id,player_code,first_name,second_name,web_name,team_id,position_id,season,opta_code
0,1,80201,Bernd,Leno,Leno,1,1,2021-22,NaN
1,2,115918,Rúnar Alex,Rúnarsson,Rúnarsson,1,1,2021-22,NaN
2,3,47431,Willian,Borges Da Silva,Willian,1,3,2021-22,NaN
3,4,54694,Pierre-Emerick,Aubameyang,Aubameyang,1,4,2021-22,NaN
4,5,58822,Cédric,Soares,Cédric,1,2,2021-22,NaN


In [8]:
#@title { vertical-output: true }
# --------------------------------------------------
# CHECK 1: missing player codes
# --------------------------------------------------

missing = players[players["player_code"].isna()]

print("\n=== MISSING PLAYER CODES ===")
print(f"Rows: {len(missing):,}")

display(missing.head(20))


# --------------------------------------------------
# CHECK 2: duplicate code within the SAME season
# --------------------------------------------------

duplicates = (
    players
    .groupby(["season", "player_code"])
    .size()
    .reset_index(name="rows")
    .query("rows > 1")
)

print("\n=== DUPLICATE CODE WITHIN SEASON ===")
print(f"Cases: {len(duplicates):,}")

display(duplicates.head(20))


# --------------------------------------------------
# CHECK 3: names associated with each code
# --------------------------------------------------

name_check = (
    players
    .groupby("player_code")
    .agg(
        seasons=("season", "nunique"),
        names=("web_name", lambda x: sorted(set(x.dropna()))),
        first_names=("first_name", lambda x: sorted(set(x.dropna()))),
        second_names=("second_name", lambda x: sorted(set(x.dropna())))
    )
    .reset_index()
)

name_check["web_name_count"] = name_check["names"].apply(len)
name_check["second_name_count"] = name_check["second_names"].apply(len)

name_changes = (
    name_check[
        (name_check["web_name_count"] > 1) |
        (name_check["second_name_count"] > 1)
    ]
    .sort_values(
        ["second_name_count", "web_name_count"],
        ascending=False
    )
)

print("\n=== CODES WITH NAME CHANGES ===")
print(f"Cases: {len(name_changes):,}")

display(name_changes.head(50))


# --------------------------------------------------
# CHECK 4: players appearing across multiple seasons
# --------------------------------------------------

multi_season = (
    name_check[name_check["seasons"] > 1]
    .sort_values("seasons", ascending=False)
)

print("\n=== CROSS-SEASON PLAYERS ===")
print(f"Players: {len(multi_season):,}")

display(multi_season.head(50))


# --------------------------------------------------
# CHECK 5: team / position changes
# --------------------------------------------------

career_changes = (
    players
    .groupby("player_code")
    .agg(
        seasons=("season", "nunique"),
        teams=("team_id", "nunique"),
        positions=("position_id", "nunique")
    )
    .reset_index()
)

print("\n=== TEAM CHANGES ===")
display(
    career_changes[
        career_changes["teams"] > 1
    ].sort_values("teams", ascending=False).head(30)
)

print("\n=== POSITION CHANGES ===")
display(
    career_changes[
        career_changes["positions"] > 1
    ].sort_values("positions", ascending=False).head(30)
)


=== MISSING PLAYER CODES ===
Rows: 0


,fpl_element_id,player_code,first_name,second_name,web_name,team_id,position_id,season,opta_code



=== DUPLICATE CODE WITHIN SEASON ===
Cases: 0


,season,player_code,rows



=== CODES WITH NAME CHANGES ===
Cases: 197


...

,player_code,seasons,names,first_names,second_names,web_name_count,second_name_count
1398,514356,5,[Lavia],"[Romeo, Roméo]",[Lavia],1,1
39,40383,5,[Forster],[Fraser],[Forster],1,1
28,37096,5,[Fabianski],"[Lukasz, Łukasz]","[Fabianski, Fabiański]",1,2
1371,510362,5,[Toti],"[Toti, Toti António]",[Gomes],1,1
1345,501837,5,[Mosquera],[Yerson],"[Mosquera, Mosquera Valdelamar]",1,2
1283,493250,5,"[Amad, Diallo]",[Amad],[Diallo],2,1
1280,493105,5,[Garnacho],[Alejandro],"[Garnacho, Garnacho Ferreyra]",1,2
1251,490721,5,"[Bueno, H.Bueno]",[Hugo],"[Bueno, Bueno López]",2,2
1245,490145,5,[Scarlett],[Dane],[Scarlett],1,1
1242,490094,5,[Iroegbunam],[Tim],[Iroegbunam],1,1



=== TEAM CHANGES ===


,player_code,seasons,teams,positions
978,440323,5,5,1
839,243016,5,4,1
1110,463034,4,4,1
924,432714,5,4,1
652,215059,5,4,1
1398,514356,5,4,1
512,184349,5,4,2
405,159533,5,4,1
255,103955,4,4,1
619,209243,5,4,1



=== POSITION CHANGES ===


,player_code,seasons,teams,positions
82,55037,3,2,2
92,56983,3,2,2
158,80146,4,3,2
164,80954,2,2,2
198,87873,2,2,2
413,165153,4,2,2
447,170137,3,2,2
471,173818,2,2,2
482,174932,3,2,2
491,178186,5,1,2


In [9]:
#@title { vertical-output: true }
import re
import unicodedata

def normalise_name(value):
    if pd.isna(value):
        return ""

    value = str(value).lower().strip()

    # remove accents
    value = "".join(
        c for c in unicodedata.normalize("NFKD", value)
        if not unicodedata.combining(c)
    )

    # remove punctuation/spaces
    value = re.sub(r"[^a-z0-9]", "", value)

    return value


players["full_name"] = (
    players["first_name"].fillna("")
    + " "
    + players["second_name"].fillna("")
).str.strip()

players["full_name_normalised"] = (
    players["full_name"].apply(normalise_name)
)

normalised_name_check = (
    players
    .groupby("player_code")
    .agg(
        seasons=("season", "nunique"),
        names=("full_name", lambda x: sorted(set(x))),
        normalised_names=(
            "full_name_normalised",
            lambda x: sorted(set(x))
        )
    )
    .reset_index()
)

normalised_name_check["normalised_name_count"] = (
    normalised_name_check["normalised_names"].apply(len)
)

possible_reuse = (
    normalised_name_check[
        normalised_name_check["normalised_name_count"] > 1
    ]
    .sort_values("normalised_name_count", ascending=False)
)

print(
    f"Possible genuine name mismatches: "
    f"{len(possible_reuse):,}"
)

display(possible_reuse.head(50))

Possible genuine name mismatches: 129


,player_code,seasons,names,normalised_names,normalised_name_count
452,171314,5,"[Rúben Gato Alves Dias, Rúben Santos Gato Alve...","[rubendossantosgatoalvesdias, rubengatoalvesdi...",3
697,220566,5,"[Rodrigo 'Rodri' Hernandez, Rodrigo 'Rodri' He...","[rodrigohernandez, rodrigorodrihernandez, rodr...",3
321,121145,4,"[João Cancelo, João Cavaco Cancelo, João Pedro...","[joaocancelo, joaocavacocancelo, joaopedrocava...",3
28,37096,5,"[Lukasz Fabianski, Łukasz Fabiański]","[lukaszfabianski, ukaszfabianski]",2
74,51940,2,"[David De Gea Quintana, David de Gea]","[daviddegea, daviddegeaquintana]",2
191,85971,5,"[Heung-Min Son, Son Heung-min]","[heungminson, sonheungmin]",2
217,92371,2,"[Pablo Marí, Pablo Marí Villar]","[pablomari, pablomarivillar]",2
128,66749,3,"[Romelu Lukaku, Romelu Lukaku Bolingoli]","[romelulukaku, romelulukakubolingoli]",2
104,58822,3,"[Cédric Alves Soares, Cédric Soares]","[cedricalvessoares, cedricsoares]",2
261,106468,4,"[Alexandre Moreno Lopera, Álex Moreno Lopera]","[alexandremorenolopera, alexmorenolopera]",2


In [ ]:
#@title { vertical-output: true }

In [ ]:
#@title { vertical-output: true }

In [ ]:
#@title { vertical-output: true }